# COPD Risk Finder: Impact Analysis

**Purpose:** Quantify the expected population-level impact of deploying the COPD Risk Finder in Spanish primary care. All inputs are documented with their source. Parameters marked `[ASSUMPTION]` are estimates that must be validated before clinical or commercial use.

---

### Assumption Registry

| ID | Parameter | Value | Basis | Flag |
|---|---|---|---|---|
| V1 | Population ≥40 years (Spain) | 28,759,538 | INE Table 68520, 2025 | Official |
| V2 | COPD prevalence (≥40 years) | 11.8% | EPISCAN II, Soriano et al., *Arch Bronconeumol* 2021, DOI: 10.1016/j.arbres.2020.07.024 | Peer-reviewed |
| V3 | Underdiagnosis rate | 74.7% | EPISCAN II (corrects challenge brief’s 70%) | Peer-reviewed |
| V4 | Model sensitivity | 88.9% | Project test set, n=1,990 (03_Models.ipynb) | Project data |
| V5 | Model specificity | 17.5% | Project test set, n=1,990 (03_Models.ipynb) | Project data |
| V6 | Spirometry cost | €47.75 | MUFACE baremo oficial 2025, BOE-A-2025-8730, Anexo 10 | Official |
| V7 | COPD hospitalisation cost | €4,339 | Ministerio de Sanidad, RAE-CMBD 2023 (CCS grupo 127) | Official |
| V8 | GP adoption rate | 25% | **[ASSUMPTION]** Illustrative mid-case scenario; no evidence base | Assumption |
| V9 | Treatment initiation rate | ~100% | Challenge brief: 1M diagnosed / 1M treated (directional) | Observation |
| A1 | GPs in Spanish primary care | 26,000 | Ministerio de Sanidad, Estadistica de Profesionales Sanitarios SNS 2023 | Official |
| A2 | Spirometry acceptance rate | 70-100% | No Spain-specific source available; shown as sensitivity range | Assumption |


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

pd.set_option('display.float_format', '{:,.1f}'.format)

In [2]:
# ============================================================
# PARAMETERS: edit here to update all downstream outputs
# [OFF] = Official source
# [MDL] = Project model test set
# [ASM] = Assumption
# ============================================================

PARAMS = {
    # Epidemiology [OFF]
    'pop_40_plus':              28_759_538,  # [OFF] INE Table 68520, 2025
    'copd_prevalence':          0.118,       # [OFF] EPISCAN II, Soriano et al. 2021
    'underdiagnosis_rate':      0.747,       # [OFF] EPISCAN II (corrects challenge brief 70%)

    # Model performance [MDL]
    'sensitivity':              0.889,       # [MDL] @ threshold 0.667
    'specificity':              0.175,       # [MDL] @ threshold 0.667

    # Costs [OFF]
    'spirometry_cost_eur':      47.75,       # [OFF] MUFACE BOE-A-2025-8730, Anexo 10
    'hospitalization_cost_eur': 4_339.4,     # [OFF] Ministerio de Sanidad RAE-CMBD 2023

    # Deployment [ASM]
    'adoption_rate':            0.25,        # [ASM] Illustrative 25% GP adoption
}

p = PARAMS

# ============================================================
# DERIVED QUANTITIES (no new assumptions)
# ============================================================

copd_total       = p['pop_40_plus'] * p['copd_prevalence']
copd_diagnosed   = copd_total * (1 - p['underdiagnosis_rate'])
copd_undiagnosed = copd_total * p['underdiagnosis_rate']
screening_pop    = p['pop_40_plus'] - copd_diagnosed
eff_prev         = copd_undiagnosed / screening_pop

sens = p['sensitivity']
spec = p['specificity']

# Per 10,000 screened; used in cost and summary sections
N             = 10_000
tp            = N * eff_prev * sens
fp            = N * (1 - eff_prev) * (1 - spec)
total_flagged = tp + fp
nncr          = total_flagged / tp  # referrals per confirmed diagnosis

print('=== KEY DERIVED QUANTITIES ===')
print(f'  COPD total (Spain, >=40):        {copd_total:>12,.0f}')
print(f'  COPD diagnosed:                  {copd_diagnosed:>12,.0f}')
print(f'  COPD undiagnosed:                {copd_undiagnosed:>12,.0f}')
print(f'  Eligible screening population:   {screening_pop:>12,.0f}')
print(f'  Effective prevalence (derived):  {eff_prev:>12.1%}')
print(f'  Referrals per confirmed Dx:      {nncr:>12.1f}')


=== KEY DERIVED QUANTITIES ===
  COPD total (Spain, >=40):           3,393,625
  COPD diagnosed:                       858,587
  COPD undiagnosed:                   2,535,038
  Eligible screening population:     27,900,951
  Effective prevalence (derived):          9.1%
  Referrals per confirmed Dx:              10.3


---
## Section 1: The Diagnosis Gap in Spain

Starting point: how large is the problem the tool is trying to solve?

*Sources: INE Table 68520 (2025); EPISCAN II, Soriano JB et al., Arch Bronconeumol 2021.*

In [3]:
funnel_data = {
    'Stage': [
        'Spain population ≥40 (INE 2025)',
        'Estimated COPD cases (11.8%, EPISCAN II)',
        'Currently diagnosed (~25.3%)',
        'Undiagnosed COPD: the gap (74.7%, EPISCAN II)',
    ],
    'Patients': [
        p['pop_40_plus'],
        copd_total,
        copd_diagnosed,
        copd_undiagnosed,
    ],
    'Color': ['#4A90D9', '#E8A838', '#5BAD6F', '#D94A4A']
}

df_funnel = pd.DataFrame(funnel_data)

fig1 = go.Figure(go.Bar(
    x=df_funnel['Patients'],
    y=df_funnel['Stage'],
    orientation='h',
    marker_color=df_funnel['Color'],
    text=[f'{v/1e6:.2f}M' for v in df_funnel['Patients']],
    textposition='outside',
    textfont=dict(size=13),
))

fig1.update_layout(
    title=dict(text='COPD Diagnosis Gap in Spain', font=dict(size=18)),
    xaxis_title='Number of patients',
    yaxis=dict(autorange='reversed', tickfont=dict(size=12)),
    xaxis=dict(range=[0, p['pop_40_plus'] * 1.15]),
    height=320,
    margin=dict(l=10, r=80, t=50, b=40),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig1.update_xaxes(showgrid=True, gridcolor='#EFEFEF')
fig1.show()

---
## Section 2: Impact by Risk Tier

The app stratifies patients into three tiers using the calibrated probability score:

| Tier | Probability | Action in app |
|---|---|---|
| **High** | >= 0.667 (model threshold) | Immediate spirometry referral |
| **Moderate** | 0.35 - 0.667 | CAT/MRC questionnaire, then reassess |
| **Low** | < 0.35 | No action, monitor |

> **Note on prevalence:** This section evaluates tier performance on the **challenge test set** (~76% COPD+ prevalence), which reflects the spirometry-selected population provided for this project. Counts scaled to per-10,000 should be interpreted in that context, not as real-world primary-care figures.

This section breaks the model output down by tier to show where diagnoses actually come from and compares two referral strategies.

*Metrics computed on the held-out test set (n=1,990), using the same 80/20 stratified split as `03_Models.ipynb` (random_state=42).*


In [4]:
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE = Path('..')

model_data    = joblib.load(BASE / 'models' / 'copd_model_v2.pkl')
clf           = model_data['model']
feature_names = model_data['feature_names']
HIGH_THRESH   = float(model_data['threshold'])  # 0.667, from app.py
MOD_THRESH    = 0.35                            # from app.py

features_df = pd.read_csv(BASE / 'data' / 'processed' / 'features.csv')
labels_df   = pd.read_csv(BASE / 'data' / 'processed' / 'labels.csv')
df_all      = features_df.merge(labels_df, on='id_paciente')

X = df_all[feature_names]
y = df_all['copd_label']

# Reproduce the exact same split used in 03_Models.ipynb
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Pass .values so the pipeline (fitted on arrays) does not emit a feature-name warning
y_prob = clf.predict_proba(X_test.values)[:, 1]

# Vectorized tier assignment; avoids row-wise apply
results = pd.DataFrame({
    'prob':   y_prob,
    'tier':   pd.cut(
        y_prob,
        bins=[-np.inf, MOD_THRESH, HIGH_THRESH, np.inf],
        labels=['Low', 'Moderate', 'High']
    ).astype(str),
    'actual': y_test.values
})

n_total      = len(results)
n_copd_total = int((results['actual'] == 1).sum())

tier_order  = ['High', 'Moderate', 'Low']
tier_colors = {'High': '#D94A4A', 'Moderate': '#E8A838', 'Low': '#5BAD6F'}
rows = []

for tier in tier_order:
    sub        = results[results['tier'] == tier]
    n          = len(sub)
    n_copd     = int((sub['actual'] == 1).sum())
    n_no_copd  = int((sub['actual'] == 0).sum())
    ppv        = n_copd / n if n > 0 else 0
    sens_share = n_copd / n_copd_total

    rows.append({
        'Tier':                    tier,
        'N in tier':               n,
        '% of screened':           f'{n/n_total:.1%}',
        'COPD+ (TP)':              n_copd,
        'Non-COPD (FP)':           n_no_copd,
        'PPV':                     f'{ppv:.1%}',
        '% of all COPD+ captured': f'{sens_share:.1%}',
        '_n':      n,
        '_n_copd': n_copd,
        '_ppv':    ppv,
    })

df_tiers = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in rows])
display(df_tiers)


,Tier,N in tier,% of screened,COPD+ (TP),Non-COPD (FP),PPV,% of all COPD+ captured
0,High,1739,87.4%,1351,388,77.7%,88.9%
1,Moderate,251,12.6%,169,82,67.3%,11.1%
2,Low,0,0.0%,0,0,0.0%,0.0%


In [5]:
# N = 10_000, defined in PARAMS (c3d4e5f6); required for scaling
scale = N / n_total  # test-set fractions → per-10,000 (challenge dataset prevalence ~76%)

tier_raw = {r['Tier']: r for r in rows}

high_n    = tier_raw['High']['_n'] * scale
high_dx   = tier_raw['High']['_n_copd'] * scale
high_fp   = high_n - high_dx

mod_n     = tier_raw['Moderate']['_n'] * scale
mod_dx    = tier_raw['Moderate']['_n_copd'] * scale
mod_fp    = mod_n - mod_dx

low_missed = tier_raw['Low']['_n_copd'] * scale

# Strategy A: refer High tier only
# Strategy B: refer High + Moderate
strat_labels = [
    'Strategy A<br><b>High tier only</b>',
    'Strategy B<br><b>High + Moderate</b>',
]
strat_dx   = [high_dx,              high_dx + mod_dx]
strat_fp   = [high_fp,              high_fp + mod_fp]
strat_refs = [high_n,               high_n + mod_n]
strat_ppv  = [
    high_dx / high_n if high_n > 0 else 0,
    (high_dx + mod_dx) / (high_n + mod_n) if (high_n + mod_n) > 0 else 0,
]

fig5 = go.Figure()

fig5.add_trace(go.Bar(
    name='New diagnoses found',
    x=strat_labels,
    y=strat_dx,
    marker_color=['#5BAD6F', '#3A8A52'],
    text=[f'{v:,.0f} Dx<br>PPV {ppv_val:.1%}' for v, ppv_val in zip(strat_dx, strat_ppv)],
    textposition='inside',
    textfont=dict(size=13, color='white'),
))

fig5.add_trace(go.Bar(
    name='Unnecessary referrals (FP)',
    x=strat_labels,
    y=strat_fp,
    marker_color=['#E8A838', '#C47B1A'],
    text=[f'{v:,.0f} FP' for v in strat_fp],
    textposition='inside',
    textfont=dict(size=13, color='white'),
))

fig5.update_layout(
    barmode='stack',
    title=dict(text='Referral Strategies: New Diagnoses vs Unnecessary Referrals (per 10,000)', font=dict(size=16)),
    yaxis_title='Patients referred for spirometry',
    height=440,
    margin=dict(l=10, r=20, t=60, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(x=0.62, y=0.95),
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(range=[0, (high_n + mod_n) * 1.15]),
)
fig5.update_yaxes(showgrid=True, gridcolor='#EFEFEF')
fig5.show()

# --- PPV per referred tier (High and Moderate only) ---
# Low tier is excluded: patients are not referred, so PPV does not apply.
# Low tier impact is shown as a missed-cases annotation instead.
referred_tiers  = ['High', 'Moderate']
referred_colors = [tier_colors[t] for t in referred_tiers]
ppv_vals        = [r['_ppv'] for r in rows if r['Tier'] in referred_tiers]

fig6 = go.Figure(go.Bar(
    x=referred_tiers,
    y=[v * 100 for v in ppv_vals],
    marker_color=referred_colors,
    text=[f'{v:.1%}' for v in ppv_vals],
    textposition='outside',
    textfont=dict(size=14),
))

# Reference line: dataset COPD+ rate (= PPV of a random referral, not an overall PPV)
dataset_pos_rate = n_copd_total / n_total
fig6.add_hline(
    y=dataset_pos_rate * 100, line_dash='dash', line_color='#888',
    annotation_text=f'Dataset COPD+ rate {dataset_pos_rate:.1%}',
    annotation_position='bottom right',
    annotation_font_size=11,
)

fig6.add_annotation(
    x=1.05, y=max(ppv_vals) * 100 * 0.5,
    xref='paper', yref='y',
    text=f'Low tier:<br>{low_missed:,.0f} COPD+<br>not referred<br>(missed)',
    showarrow=False,
    font=dict(size=11, color='#5BAD6F'),
    align='left',
    bgcolor='#F5F5F5',
    bordercolor='#5BAD6F',
    borderwidth=1,
)

fig6.update_layout(
    title=dict(text='PPV by Risk Tier (referred tiers only)', font=dict(size=16)),
    yaxis_title='PPV (%)',
    height=380,
    margin=dict(l=10, r=160, t=55, b=40),
    plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(range=[0, max(ppv_vals) * 130]),
)
fig6.update_yaxes(showgrid=True, gridcolor='#EFEFEF')
fig6.show()

print(f'Per 10,000 screened:')
for label, dx, fp, refs, ppv_val in zip(strat_labels, strat_dx, strat_fp, strat_refs, strat_ppv):
    clean = label.replace('<br>', ' ').replace('<b>', '').replace('</b>', '')
    print(f'  {clean}: {refs:,.0f} referred -> {dx:,.0f} new Dx, {fp:,.0f} FP  (PPV {ppv_val:.1%})')
print(f'  Low tier: {low_missed:,.0f} COPD+ cases missed')


Per 10,000 screened:
  Strategy A High tier only: 8,739 referred -> 6,789 new Dx, 1,950 FP  (PPV 77.7%)
  Strategy B High + Moderate: 10,000 referred -> 7,638 new Dx, 2,362 FP  (PPV 76.4%)
  Low tier: 0 COPD+ cases missed


---
## Section 3: National Impact at 25% GP Adoption

Scaling to Spain: if 25% of the eligible screening population is reached by the tool.

> **[ASSUMPTION V8]** The 25% adoption rate is illustrative. It represents a plausible 3-5 year deployment target but has no evidence base. Change `adoption_rate` in PARAMS to explore other scenarios.


In [6]:
patients_covered       = screening_pop * p['adoption_rate']
new_diagnoses_national = patients_covered * eff_prev * sens
total_referrals_nat    = patients_covered * (eff_prev * sens + (1 - eff_prev) * (1 - spec))
gap_closed_pct         = new_diagnoses_national / copd_undiagnosed

print(f'At {p["adoption_rate"]:.0%} adoption:')
print(f'  Patients covered:               {patients_covered:>12,.0f}')
print(f'  Total spirometry referrals:     {total_referrals_nat:>12,.0f}')
print(f'  New COPD diagnoses per year:    {new_diagnoses_national:>12,.0f}')
print(f'  % of diagnosis gap closed:      {gap_closed_pct:>12.1%}')
print()
print(f'  Total undiagnosed COPD (Spain): {copd_undiagnosed:>12,.0f}')
print(f'  Remaining undiagnosed after:    {copd_undiagnosed - new_diagnoses_national:>12,.0f}')


At 25% adoption:
  Patients covered:                  6,975,238
  Total spirometry referrals:        5,795,132
  New COPD diagnoses per year:         563,412
  % of diagnosis gap closed:             22.2%

  Total undiagnosed COPD (Spain):    2,535,038
  Remaining undiagnosed after:       1,971,626


In [7]:
# Scenario comparison: 10%, 25%, 50% adoption
scenarios = [0.10, 0.25, 0.50]
scenario_labels = ['10% adoption', '25% adoption\n(base case)', '50% adoption']
scenario_dx = [screening_pop * a * eff_prev * sens for a in scenarios]
scenario_gap = [dx / copd_undiagnosed * 100 for dx in scenario_dx]

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='New diagnoses',
    x=scenario_labels,
    y=[dx / 1000 for dx in scenario_dx],
    marker_color=['#A8C8E8', '#4A90D9', '#1A5FA8'],
    text=[f'{dx/1000:,.0f}K<br>({gap:.1f}% of gap)' for dx, gap in zip(scenario_dx, scenario_gap)],
    textposition='outside',
    textfont=dict(size=12),
))

fig4.update_layout(
    title=dict(text='Estimated New COPD Diagnoses per Year by Adoption Scenario', font=dict(size=17)),
    yaxis_title='New diagnoses (thousands)',
    height=400,
    margin=dict(l=10, r=20, t=55, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(range=[0, max(scenario_dx) / 1000 * 1.25]),
    xaxis=dict(tickfont=dict(size=12)),
)
fig4.update_yaxes(showgrid=True, gridcolor='#EFEFEF')
fig4.show()


---
### Section 3a: Treatment Pipeline

New COPD diagnoses translate directly to treatment initiations. The challenge brief reports approximately 1M diagnosed and 1M treated COPD patients in Spain, suggesting near-complete treatment uptake among confirmed cases. This is consistent with GOLD 2024 guidance, which recommends pharmacological therapy for all confirmed COPD diagnoses regardless of severity stage.

> **[V9]** Treatment initiation rate is a directional estimate derived from the challenge brief (1M diagnosed / 1M treated). No independent Spanish-specific source is available in the project data.


In [8]:
# Treatment initiation: derived from challenge brief (1M diagnosed ~= 1M treated in Spain)
TREATMENT_INIT_RATE = 1.00  # [V9] directional; challenge brief observation

adoption_scenarios_v2 = [('10%', 0.10), ('25% (base)', 0.25), ('50%', 0.50)]

print('=== TREATMENT PIPELINE ===')
print(f'Treatment initiation rate: {TREATMENT_INIT_RATE:.0%} (challenge brief observation)')
print()
hdr = f"{'Adoption':>12} | {'New Dx/year':>14} | {'New patients on therapy/year':>30}"
print(hdr)
print('-' * len(hdr))
for label, rate in adoption_scenarios_v2:
    dx = screening_pop * rate * eff_prev * sens
    on_therapy = dx * TREATMENT_INIT_RATE
    print(f'{label:>12} | {dx:>14,.0f} | {on_therapy:>30,.0f}')


=== TREATMENT PIPELINE ===
Treatment initiation rate: 100% (challenge brief observation)

    Adoption |    New Dx/year |   New patients on therapy/year
--------------------------------------------------------------
         10% |        225,365 |                        225,365
  25% (base) |        563,412 |                        563,412
         50% |      1,126,824 |                      1,126,824


---
### Section 3b: Referral Capacity per GP Practice

Translating the national figures to the individual GP level makes the adoption barrier concrete: how many additional spirometry referrals would a GP need to generate per week?

> **[A1]** Spain has approximately 26,000 GPs in primary care. Source: Ministerio de Sanidad, Estadistica de Profesionales Sanitarios SNS 2023.

> **Prevalence note:** model sensitivity and specificity were measured on the challenge test set (~76% COPD+ prevalence). The referral estimates here apply these rates to real-world prevalence (9.1%), assuming prevalence-independence of the classifier. This may overestimate referral volume if performance degrades at lower prevalence.


In [9]:
N_GPS_SPAIN = 26_000  # [A1] Ministerio de Sanidad, Estadistica de Profesionales Sanitarios SNS 2023

eligible_per_gp = screening_pop / N_GPS_SPAIN

# Referral rate at real-world effective prevalence
referral_rate_rw = eff_prev * sens + (1 - eff_prev) * (1 - spec)
dx_rate_rw       = eff_prev * sens

print(f'Eligible patients per GP: {eligible_per_gp:,.0f} (screening pop / {N_GPS_SPAIN:,} GPs)')
print(f'Real-world effective prevalence: {eff_prev:.1%}')
print()
hdr = (
    f"{'Adoption':>10} | {'Pts screened/GP':>15} | "
    f"{'Referrals/GP/yr':>16} | {'New Dx/GP/yr':>13} | {'Referrals/wk':>13}"
)
print(hdr)
print('-' * len(hdr))
for label, rate in [('10%', 0.10), ('25%', 0.25), ('50%', 0.50)]:
    screened  = eligible_per_gp * rate
    referrals = screened * referral_rate_rw
    new_dx    = screened * dx_rate_rw
    per_week  = referrals / 52
    print(
        f'{label:>10} | {screened:>15,.0f} | '
        f'{referrals:>16,.0f} | {new_dx:>13.1f} | {per_week:>13.1f}'
    )


Eligible patients per GP: 1,073 (screening pop / 26,000 GPs)
Real-world effective prevalence: 9.1%

  Adoption | Pts screened/GP |  Referrals/GP/yr |  New Dx/GP/yr |  Referrals/wk
-------------------------------------------------------------------------------
       10% |             107 |               89 |           8.7 |           1.7
       25% |             268 |              223 |          21.7 |           4.3
       50% |             537 |              446 |          43.3 |           8.6


---
### Section 3c: Sensitivity Analysis

Two parameters carry meaningful real-world uncertainty beyond the model itself:

- **GP adoption rate**: how widely the tool is deployed across primary care
- **Spirometry acceptance rate**: the fraction of flagged patients who actually attend their referral appointment

> **[A2]** No Spain-specific data on spirometry referral compliance is available in the project. The 70% and 85% scenarios are illustrative bounds. A 100% acceptance rate is the current model assumption and represents an upper bound.

Key insight: cost per confirmed diagnosis is invariant to acceptance rate, because TP and FP are reduced proportionally. What changes is total diagnostic yield and total programme cost.


In [10]:
acceptance_rates  = [1.00, 0.85, 0.70]
adoption_rates_sa = [0.10, 0.25, 0.50]
acc_labels = ['100% (base)', '85% [A2]', '70% [A2]']
adp_labels = ['Adoption 10%', 'Adoption 25%', 'Adoption 50%']

col_w = 18
hdr = f"{'Scenario':>{col_w}} | " + " | ".join(f"{l:>{col_w}}" for l in acc_labels)
sep = '-' * len(hdr)

# Grid: new diagnoses per year
print('New Diagnoses per Year')
print(hdr)
print(sep)
for label, ad_rate in zip(adp_labels, adoption_rates_sa):
    base_dx = screening_pop * ad_rate * eff_prev * sens
    cells   = [f"{base_dx * acc:>{col_w},.0f}" for acc in acceptance_rates]
    print(f"{label:>{col_w}} | " + " | ".join(cells))

print()

# Grid: total programme cost (EUR millions)
print('Total Programme Cost (EUR millions)')
print(hdr)
print(sep)
for label, ad_rate in zip(adp_labels, adoption_rates_sa):
    base_refs = screening_pop * ad_rate * (eff_prev * sens + (1 - eff_prev) * (1 - spec))
    cells = [f"EUR {base_refs * acc * p['spirometry_cost_eur'] / 1e6:>{col_w - 4},.1f}M" for acc in acceptance_rates]
    print(f"{label:>{col_w}} | " + " | ".join(cells))

print()
print(f'Cost per new diagnosis: EUR {nncr * p["spirometry_cost_eur"]:,.0f} (invariant to acceptance rate)')

# Grouped bar: new diagnoses by scenario
bar_colors = ['#4A90D9', '#7EB3E8', '#B5D4F5']
fig_sa = go.Figure()
for acc, lbl, col in zip(acceptance_rates, acc_labels, bar_colors):
    dx_vals = [screening_pop * ad * eff_prev * sens * acc / 1e3 for ad in adoption_rates_sa]
    fig_sa.add_trace(go.Bar(
        name=f'Accept {lbl}',
        x=adp_labels,
        y=dx_vals,
        marker_color=col,
        text=[f'{v:,.0f}K' for v in dx_vals],
        textposition='outside',
        textfont=dict(size=11),
    ))
fig_sa.update_layout(
    barmode='group',
    title=dict(text='New Diagnoses per Year: Adoption Rate x Spirometry Acceptance Rate', font=dict(size=15)),
    yaxis_title='New diagnoses (thousands)',
    height=420,
    margin=dict(l=10, r=20, t=55, b=60),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(x=0.01, y=0.97),
    xaxis=dict(tickfont=dict(size=12)),
)
fig_sa.update_yaxes(showgrid=True, gridcolor='#EFEFEF')
fig_sa.show()


New Diagnoses per Year
          Scenario |        100% (base) |           85% [A2] |           70% [A2]
---------------------------------------------------------------------------------
      Adoption 10% |            225,365 |            191,560 |            157,755
      Adoption 25% |            563,412 |            478,900 |            394,389
      Adoption 50% |          1,126,824 |            957,801 |            788,777

Total Programme Cost (EUR millions)
          Scenario |        100% (base) |           85% [A2] |           70% [A2]
---------------------------------------------------------------------------------
      Adoption 10% | EUR          110.7M | EUR           94.1M | EUR           77.5M
      Adoption 25% | EUR          276.7M | EUR          235.2M | EUR          193.7M
      Adoption 50% | EUR          553.4M | EUR          470.4M | EUR          387.4M

Cost per new diagnosis: EUR 491 (invariant to acceptance rate)


---
## Section 4: Cost Estimate

**Screening cost:** each flagged patient is referred for spirometry. Cost per spirometry = 47.75 EUR (MUFACE BOE-A-2025-8730, Anexo 10). Note this is a reimbursement ceiling, not a direct SNS production cost. Treat it as a lower-bound proxy.

**Break-even:** how many COPD hospitalisations must be prevented per newly diagnosed patient for the programme to be cost-neutral? Based on 4,339 EUR average hospitalisation cost (Ministerio de Sanidad RAE-CMBD 2023, CCS grupo 127).


In [11]:
# Cost per new confirmed diagnosis
cost_per_referral  = p['spirometry_cost_eur']
cost_per_new_dx    = cost_per_referral * nncr

# National screening cost (25% adoption)
total_cost_national = total_referrals_nat * cost_per_referral

# Break-even: hospitalisations prevented per new Dx to recover screening cost
hosp_per_dx_breakeven = cost_per_new_dx / p['hospitalization_cost_eur']

print('=== COST ANALYSIS ===')
print(f'  Cost per spirometry referral:        €{cost_per_referral:>8.2f}')
print(f'  Referrals per confirmed new Dx:      {nncr:>8.1f}')
print(f'  Cost to find one new Dx:             €{cost_per_new_dx:>8.2f}')
print()
print(f'  Avg COPD hospitalisation cost:       €{p["hospitalization_cost_eur"]:>8,.1f}')
print(f'  Hospitalisations to break even:      {hosp_per_dx_breakeven:>8.2f} per new Dx')
print()
print(f'  National screening cost (25% adopt): €{total_cost_national/1e6:>8,.1f}M')
print(f'  National new diagnoses:                {new_diagnoses_national:>8,.0f}')
print()
print(f'  => Finding one undiagnosed COPD patient requires {nncr:.0f} spirometry referrals.')
print(f'     At €{cost_per_referral} per test, cost = €{cost_per_new_dx:,.0f} per new diagnosis.')
print(f'     This is recovered if early treatment prevents {hosp_per_dx_breakeven:.2f} hospitalisations')
print(f'     over the patient life (each worth €{p["hospitalization_cost_eur"]:,}).')


=== COST ANALYSIS ===
  Cost per spirometry referral:        €   47.75
  Referrals per confirmed new Dx:          10.3
  Cost to find one new Dx:             €  491.15

  Avg COPD hospitalisation cost:       € 4,339.4
  Hospitalisations to break even:          0.11 per new Dx

  National screening cost (25% adopt): €   276.7M
  National new diagnoses:                 563,412

  => Finding one undiagnosed COPD patient requires 10 spirometry referrals.
     At €47.75 per test, cost = €491 per new diagnosis.
     This is recovered if early treatment prevents 0.11 hospitalisations
     over the patient life (each worth €4,339.4).


---
## Section 5: Summary

All key outputs in one view.


In [12]:
summary = {
    'Metric': [
        'Undiagnosed COPD in Spain (EPISCAN II + INE)',
        'Eligible screening population',
        'Model sensitivity (test set)',
        'Model specificity (test set)',
        'Referrals per confirmed diagnosis',
        'Miss rate (undiagnosed COPD not flagged)',
        'New diagnoses per year (25% adoption)',
        'Diagnosis gap closed (25% adoption)',
        'Total spirometry referrals (25% adoption)',
        'Cost per confirmed new diagnosis',
        'Hospitalisations to break even per new Dx',
    ],
    'Value': [
        f'{copd_undiagnosed/1e6:.2f}M',
        f'{screening_pop/1e6:.1f}M',
        f'{sens:.1%}',
        f'{spec:.1%}',
        f'{nncr:.1f}',
        f'{(1 - sens):.1%}',
        f'{new_diagnoses_national:,.0f}',
        f'{gap_closed_pct:.1%}',
        f'{total_referrals_nat:,.0f}',
        f'€{cost_per_new_dx:,.0f}',
        f'{hosp_per_dx_breakeven:.2f}',
    ],
    'Source': [
        'EPISCAN II 2021 + INE 2025',
        'EPISCAN II 2021 + INE 2025',
        'Model test set (03_Models.ipynb)',
        'Model test set (03_Models.ipynb)',
        'Derived from test set',
        'Model test set',
        'Model + EPISCAN II + [ASSUMPTION: 25% adoption]',
        'Model + EPISCAN II + [ASSUMPTION: 25% adoption]',
        'Model + EPISCAN II + [ASSUMPTION: 25% adoption]',
        'Derived + MUFACE BOE 2025',
        'Derived + Ministerio de Sanidad RAE-CMBD 2023',
    ]
}

df_summary = pd.DataFrame(summary)
display(df_summary)


,Metric,Value,Source
0,Undiagnosed COPD in Spain (EPISCAN II + INE),2.54M,EPISCAN II 2021 + INE 2025
1,Eligible screening population,27.9M,EPISCAN II 2021 + INE 2025
2,Model sensitivity (test set),88.9%,Model test set (03_Models.ipynb)
3,Model specificity (test set),17.5%,Model test set (03_Models.ipynb)
4,Referrals per confirmed diagnosis,10.3,Derived from test set
5,Miss rate (undiagnosed COPD not flagged),11.1%,Model test set
6,New diagnoses per year (25% adoption),"563,412",Model + EPISCAN II + [ASSUMPTION: 25% adoption]
7,Diagnosis gap closed (25% adoption),22.2%,Model + EPISCAN II + [ASSUMPTION: 25% adoption]
8,Total spirometry referrals (25% adoption),"5,795,132",Model + EPISCAN II + [ASSUMPTION: 25% adoption]
9,Cost per confirmed new diagnosis,€491,Derived + MUFACE BOE 2025


---

### Interpretation notes

1. **High miss rate (11.1%)** is the unavoidable trade-off of the high-sensitivity threshold. The model is tuned to find as many undiagnosed cases as possible, accepting that some will be missed. The Low tier captures these patients for monitoring rather than discarding them.

2. **Referral volume per confirmed diagnosis** is the key operational metric for GP adoption. The tier comparison in Section 2 shows how Strategy A (High only) vs Strategy B (High + Moderate) shifts this ratio, allowing GPs to choose based on their capacity.

3. **Break-even at 0.11 hospitalisations per new diagnosis** is highly achievable. GOLD 2024 pharmacotherapy data show that LAMA therapy reduces moderate-to-severe exacerbations by ~20-30% in newly diagnosed patients. For a patient with even one hospitalisation in their clinical history, early treatment is likely to pay for the screening cost within 1-2 years.

4. **All cost figures are subject to the spirometry cost proxy** (MUFACE baremo, 47.75 EUR). This is a reimbursement ceiling rather than a direct SNS production cost. If actual cost is lower, cost-per-diagnosis improves proportionally.

5. **The 25% adoption scenario is illustrative.** Sensitivity analysis: at 10% adoption -> ~225K new diagnoses/year; at 50% -> ~1.13M/year.


---
### Limitations & Future Improvements

**1. Prevalence gap (real-world deployment)**  
The figures above reflect the challenge dataset's prevalence distribution (~76% COPD positive). In a real primary-care setting, undiagnosed COPD prevalence among eligible patients is approximately **9.1%** (EPISCAN II + INE 2025). At that prevalence, PPV would drop substantially and unnecessary referrals would increase proportionally. Any production deployment should include a Bayesian prevalence correction to set realistic expectations for GP adoption.

**2. Reducing false positives with the patient questionnaire**  
The current model relies on structured EHR features (demographics, comorbidities, medications). The **patient questionnaire** integrated in the app collects additional clinical signals: smoking history, occupational exposure, and symptom burden (dyspnoea, chronic cough, sputum). These are direct COPD risk indicators not available at model training time.
Incorporating questionnaire responses as model features would be expected to **increase specificity**, directly reducing the false positive burden in the Moderate tier. This is particularly valuable for Strategy B: a second-pass filter on questionnaire score could retain most of the true diagnoses while cutting unnecessary spirometry referrals.
